# 10 · Function Calling

**Idea:** the model can't do math or know the time reliably — but it *can* ask you to run a tool. You expose tools (JSON schemas), the model returns a tool call, you execute it, then feed the result back. This is the heart of agents.

Uses `study_buddy/tools.py` (calculator + current_time). `qwen3-coder:30b` supports tool calling.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'study_buddy').exists())
sys.path.insert(0, str(root))

from study_buddy import get_provider
from study_buddy.tools import TOOL_SCHEMAS, dispatch
llm = get_provider('ollama')  # or 'openai'

In [ ]:
messages = [{'role': 'user', 'content': 'What is 47 * 89, and what time is it now?'}]
resp = llm.chat(messages, tools=TOOL_SCHEMAS, temperature=0)
print('tool calls requested:')
for call in resp.tool_calls:
    print('  ', call['name'], call['arguments'])

In [ ]:
# Execute each requested tool and collect observations.
for call in resp.tool_calls:
    result = dispatch(call['name'], call['arguments'])
    print(f"{call['name']}({call['arguments']}) = {result}")

In [ ]:
# Feed results back so the model can compose a final natural-language answer.
followup = messages + [
    {'role': 'assistant', 'content': '', 'tool_calls': [
        {'type': 'function', 'function': {'name': c['name'], 'arguments': c['arguments']}}
        for c in resp.tool_calls]},
]
for c in resp.tool_calls:
    followup.append({'role': 'tool', 'content': dispatch(c['name'], c['arguments']), 'name': c['name']})

final = llm.chat(followup, temperature=0)
print(final.text)

**Takeaway:** tool calling = model decides *which* function to run with *what* arguments; your code executes it. Loop this and you have an agent.

Next → `11_structured_output.ipynb`